# PRFT
Product Right First Time. The products with a released / pass status at the time of initial process (Without rework). 100% running properly as well from material supply to end product / goods.


In [ ]:
import pandas as pd
import numpy as np

monthDict = {'January':1,'February':2,'March':3, 'April':4, 'May':5, 'June':6, 'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}

df=pd.read_excel("../Dataset/prod_output.xlsx",header=0)
dfA=pd.DataFrame(df)
dfB=dfA.reset_index()
dfB['Opr_daily'] = dfB['Opr_daily'].fillna(0)
dfB['ProcessDate'] = (pd.to_datetime(dfB['ProcessDate'])).dt.strftime('%d-%m-%Y')
dfB['month']=(pd.to_datetime(dfB['ProcessDate'])).dt.strftime('%B')
section=["COLORING","TUBING","STRANDING","JACKETING","ARMORING"]

# Standarizing the data.
prodStat=["released","not-released","rework","direct-pass"]
# Set rework and direct-pass as a not-released product
dfC=dfB[~dfB['RepoStatus'].isin(['Mc-off', 'Cancel', 'Hold'])]
directPassToNotReleased=dfB[dfB['RepoStatus']=="Direct-pass"]['RepoStatus'].unique()
dfC['RepoStatus']=dfC.apply(lambda row:'Not-released' if row['RepoStatus'] in directPassToNotReleased else row['RepoStatus'],axis=1)
reworkToNotreleased=dfC[dfC['RepoStatus']=="Rework"]['RepoStatus'].unique()
dfC['RepoStatus']=dfC.apply(lambda row:'Not-released' if row['RepoStatus'] in reworkToNotreleased else row['RepoStatus'],axis=1)
pendingToNotreleased=dfC[dfC['RepoStatus']=="Pending"]['RepoStatus'].unique()
dfC['RepoStatus']=dfC.apply(lambda row:'Not-released' if row['RepoStatus'] in pendingToNotreleased else row['RepoStatus'],axis=1)

prftA = pd.MultiIndex.from_product([section, monthDict], names=['GroupOrMachine', 'month'])
prftB = pd.DataFrame(index=prftA).reset_index()
prftC = prftB.merge(dfC, on=['GroupOrMachine', 'month'], how='left', indicator=True)
prftC['total'] = (prftC['_merge'] == 'both').astype(int)
prftC.drop(columns=['_merge'], inplace=True)
prftD = prftC.groupby(['GroupOrMachine', 'month'])['Qty_CTN'].sum().reset_index(name="prodOutputTotalInCtn")

print(dfB)